In [12]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

load_dotenv('D:/Semantic-Ecommerce-Recommender/.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

df = pd.read_csv('D:/Semantic-Ecommerce-Recommender/data/processed/amazon_featured.csv')
print(f"Dataset shape: {df.shape}")
print(f"API Key loaded: {bool(GROQ_API_KEY)}")

Dataset shape: (8000, 20)
API Key loaded: True


In [13]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

# Test embedding
test_embedding = model.encode("laptop for video editing")
print(f"Embedding dimension: {len(test_embedding)}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5514.24it/s]


Model loaded!
Embedding dimension: 384


In [14]:
# Create rich text for each product to embed
def create_product_text(row):
    return f"{row['title']} | Category: {row['category_name']} | Rating: {row['stars']} stars | Price: ${row['price']}"

df['product_text'] = df.apply(create_product_text, axis=1)

print("Sample product texts:")
for i in range(3):
    print(f"\n{i+1}. {df['product_text'].iloc[i]}")

Sample product texts:

1. 340-Piece Drywall Anchors and Screws Combo Pack - Includes 170 Plastic Wall Anchors and 170 Screws - Assorted Sizes - Organizer Box Included | Category: Fasteners | Rating: 4.6 stars | Price: $6.9

2. Girl Tutu Skirts,3Layers Tulle Sequin Star Ballet Dance Tutu Skirt Princess Christmas Skirt for Girl Toddler 2-8 Years | Category: Girls' Clothing | Rating: 4.3 stars | Price: $7.99

3. Corsair iCUE 5000X RGB Tempered Glass Mid-Tower ATX PC Smart Case - White | Category: Computer Components | Rating: 4.8 stars | Price: $189.99


In [15]:
print("Generating embeddings for all 8000 products...")
print("This may take 2-3 minutes...")

product_texts = df['product_text'].tolist()
embeddings = model.encode(
    product_texts,
    batch_size=64,
    show_progress_bar=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print("Embeddings generated successfully!")

# Save embeddings
np.save('D:/Semantic-Ecommerce-Recommender/models/product_embeddings.npy', embeddings)
print("Embeddings saved!")

Generating embeddings for all 8000 products...
This may take 2-3 minutes...


Batches: 100%|██████████| 125/125 [01:09<00:00,  1.81it/s]


Embeddings shape: (8000, 384)
Embeddings generated successfully!
Embeddings saved!


In [16]:
import faiss

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)
index.add(embeddings.astype('float32'))

print(f"FAISS index built!")
print(f"Total vectors indexed: {index.ntotal}")
print(f"Embedding dimension: {dimension}")

# Save FAISS index
faiss.write_index(index, 'D:/Semantic-Ecommerce-Recommender/models/faiss_index.bin')
print("FAISS index saved!")

FAISS index built!
Total vectors indexed: 8000
Embedding dimension: 384
FAISS index saved!


In [17]:
def semantic_search(query, top_n=5):
    """Search products using semantic similarity."""
    
    # Encode the query
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS index
    scores, indices = index.search(query_embedding.astype('float32'), top_n)
    
    # Get results
    results = df.iloc[indices[0]].copy()
    results['semantic_score'] = scores[0].round(4)
    
    return results[['title', 'category_name', 'stars', 'price', 'semantic_score']]

# Test semantic search
print("=== Query: laptop for video editing under $1000 ===")
print(semantic_search("laptop for video editing under $1000"))

print("\n=== Query: wireless noise cancelling headphones ===")
print(semantic_search("wireless noise cancelling headphones"))

print("\n=== Query: yoga mat non slip thick ===")
print(semantic_search("yoga mat non slip thick"))

=== Query: laptop for video editing under $1000 ===
                                                  title        category_name  \
422   Acer Aspire Slim Laptop, 20GB RAM 1TB SSD, 15....  Computers & Tablets   
7659  Lenovo ThinkPad X1 Nano 13" 2K(2160 x 1350) To...  Computers & Tablets   
2178  Dell Latitude 12 5000 5280 Business Laptop - 1...  Computers & Tablets   
75    Lenovo 2022 IdeaPad 1 15.6" HD Laptop, Athlon ...  Computers & Tablets   
3297  Dell Inspiron 15 3520 Laptop - 15.6-inch FHD (...  Computers & Tablets   

      stars   price  semantic_score  
422     4.0  429.99          0.6050  
7659    1.0  979.99          0.5976  
2178    4.7  189.90          0.5947  
75      4.4  349.00          0.5922  
3297    3.9  699.77          0.5647  

=== Query: wireless noise cancelling headphones ===
                                                  title         category_name  \
7525  Active Noise Cancelling Headphones E600Pro, 80...  Headphones & Earbuds   
7948  Wireless Earbuds,B

In [18]:
models = client.models.list()
for m in models.data:
    print(m.id)

meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
groq/compound-mini
openai/gpt-oss-safeguard-20b
allam-2-7b
canopylabs/orpheus-v1-english
groq/compound
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
qwen/qwen3.6-27b
openai/gpt-oss-120b
whisper-large-v3-turbo
whisper-large-v3
canopylabs/orpheus-arabic-saudi


In [25]:
# Test with available model
test = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_tokens=50
)
print(test.choices[0].message.content)

In [26]:
client = Groq(api_key=GROQ_API_KEY)

def rag_recommend(query, top_n=5):
    """Full RAG pipeline: retrieve + generate."""
    
    # RETRIEVE
    results = semantic_search(query, top_n=top_n)
    
    # Format context for LLM
    context = ""
    for i, (_, row) in enumerate(results.iterrows(), 1):
        context += f"{i}. {row['title']}\n"
        context += f"   Category: {row['category_name']}\n"
        context += f"   Rating: {row['stars']} stars | Price: ${row['price']}\n"
        context += f"   Semantic Match Score: {row['semantic_score']}\n\n"
    
    # GENERATE
    system_prompt = """You are an intelligent e-commerce shopping assistant.
    You help users find the perfect product based on their natural language queries.
    When given search results, provide a helpful 3-4 sentence response that:
    1. Acknowledges what the user is looking for
    2. Highlights the best matching product and why
    3. Mentions the price range available
    Be friendly, specific, and helpful."""
    
    user_prompt = f"""
    User query: "{query}"
    
    Retrieved products:
    {context}
    
    Please provide a helpful recommendation response based on these results.
    """
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=300
    )
    
    return {
        'query': query,
        'results': results,
        'llm_response': response.choices[0].message.content
    }

# Test RAG pipeline
output = rag_recommend("laptop for video editing under $1000")
print(f"Query: {output['query']}")
print(f"\nTop Products:")
print(output['results'].to_string(index=False))
print(f"\nAI Response:")
print(output['llm_response'])

Query: laptop for video editing under $1000

Top Products:
                                                                                                                                                                                                title       category_name  stars  price  semantic_score
   Acer Aspire Slim Laptop, 20GB RAM 1TB SSD, 15.6'' FHD Display, Intel Celeron N Series Processor, RJ-45, HDMI, Webcam, USB A&C, WiFi, Long Battery Life, Windows 11, 1-Year Microsoft 365, Mousepad Computers & Tablets    4.0 429.99          0.6050
Lenovo ThinkPad X1 Nano 13" 2K(2160 x 1350) Touch 16:10 Ultra-Light Laptop Intel Evo i7-1160G7 Intel Iris Xe Graphics WiFi 6 450nits 100% sRGB 2X Thunderbolt 4 Win11 Pro W/HDMI (16GB RAM | 1TB SSD) Computers & Tablets    1.0 979.99          0.5976
                                                 Dell Latitude 12 5000 5280 Business Laptop - 12.5in (1366x768), Intel Core i5-7200U, 256GB SSD, 16GB DDR4, Webcam, Windows 10 Professional (Renewed)

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Rebuild TF-IDF
df['combined_text'] = df['title'] + ' ' + df['category_name']
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['combined_text'])

def tfidf_search(query, top_n=5):
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_n]
    results = df.iloc[top_indices][['title', 'category_name', 'stars', 'price']].copy()
    results['tfidf_score'] = scores[top_indices].round(4)
    return results

# Compare on same query
test_query = "gaming keyboard mechanical rgb"

print(f"Query: '{test_query}'")
print("\n--- TF-IDF Results ---")
print(tfidf_search(test_query)[['title', 'category_name', 'tfidf_score']])

print("\n--- Semantic Search Results ---")
sem_results = semantic_search(test_query)
print(sem_results[['title', 'category_name', 'semantic_score']])

Query: 'gaming keyboard mechanical rgb'

--- TF-IDF Results ---
                                                  title  \
1662  X79 Wired/Wireless Bluetooth Mechanical Gaming...   
3304  RGB Wired 75% Percent Mechanical Keyboard with...   
527   RedThunder One Handed Gaming Keyboard RGB Back...   
52    MageGee Wireless Gaming Keyboard, Rechargeable...   
7920  RK ROYAL KLUDGE H81 Hot Swappable Mechanical K...   

                category_name  tfidf_score  
1662  Mac Games & Accessories       0.6436  
3304  Mac Games & Accessories       0.4941  
527               Video Games       0.4926  
52    Mac Games & Accessories       0.4909  
7920              Video Games       0.4863  

--- Semantic Search Results ---
                                                  title  \
2692  RGB Keyboard, Gaming Mechanical Keyboard with ...   
3304  RGB Wired 75% Percent Mechanical Keyboard with...   
527   RedThunder One Handed Gaming Keyboard RGB Back...   
7920  RK ROYAL KLUDGE H81 Hot Swappable Me

In [28]:
# Test RAG Q&A accuracy on 5 sample queries
test_queries = [
    "wireless bluetooth earbuds",
    "kitchen knife set sharp",
    "running shoes lightweight",
    "coffee maker with grinder",
    "laptop stand adjustable"
]

print("RAG Q&A Accuracy Test")
print("=" * 60)

for q in test_queries:
    results = semantic_search(q, top_n=3)
    top_category = results['category_name'].iloc[0]
    top_score = results['semantic_score'].iloc[0]
    print(f"\nQuery: '{q}'")
    print(f"Top match: {results['title'].iloc[0][:60]}...")
    print(f"Category: {top_category} | Score: {top_score}")

RAG Q&A Accuracy Test

Query: 'wireless bluetooth earbuds'
Top match: Wireless Earbuds Bluetooth 5.3 Headphones with Deep Bass, LE...
Category: Headphones & Earbuds | Score: 0.6754999756813049

Query: 'kitchen knife set sharp'
Top match: Carbide Burr Set,10pcs 1/8" Shank, 1/4" Head Length Tungsten...
Category: Cutting Tools | Score: 0.45170000195503235

Query: 'running shoes lightweight'
Top match: Womens Walking Running Shoes Non-Slip Athletic Tennis Breath...
Category: Women's Shoes | Score: 0.6162999868392944

Query: 'coffee maker with grinder'
Top match: Electric Salt and Pepper Grinder Set - Battery Operated Pepp...
Category: Food Service Equipment & Supplies | Score: 0.48890000581741333

Query: 'laptop stand adjustable'
Top match: Mount Plus MP-NBH-2 Laptop Mount Tray for Monitor Arms and S...
Category: Laptop Accessories | Score: 0.5234000086784363


In [29]:
import joblib

# Save the sentence transformer model path reference
config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_dimension": 384,
    "total_products_indexed": int(index.ntotal),
    "faiss_index_path": "models/faiss_index.bin",
    "embeddings_path": "models/product_embeddings.npy",
    "llm_model": "llama-3.1-8b-instant",
    "llm_provider": "Groq",
    "week": 6
}

with open('D:/Semantic-Ecommerce-Recommender/docs/rag_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("RAG config saved!")
print(json.dumps(config, indent=2))

RAG config saved!
{
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "total_products_indexed": 8000,
  "faiss_index_path": "models/faiss_index.bin",
  "embeddings_path": "models/product_embeddings.npy",
  "llm_model": "llama-3.1-8b-instant",
  "llm_provider": "Groq",
  "week": 6
}


In [32]:
# Final RAG pipeline test
output = rag_recommend("wireless bluetooth earbuds")
print(f"Query: {output['query']}")
print(f"\nTop Products:")
print(output['results'][['title', 'category_name', 'stars', 'price']].to_string(index=False))
print(f"\nAI Response:")
print(output['llm_response'])

Query: wireless bluetooth earbuds

Top Products:
                                                                                                                                                                                                title        category_name  stars  price
   Wireless Earbuds Bluetooth 5.3 Headphones with Deep Bass, LED Digital Display Charging Case, Dual Power Display, IPX5 Waterproof, Immersive Stereo Sound in-Ear Earphones with Mic for iOS Android Headphones & Earbuds    3.8  19.98
Wireless Earbuds,Bluetooth 5.3 Headphones Build in Noise Cancelling, Bluetooth Earbuds With LED Power Display, Hi-Fi Stereo, Touch Control, Waterproof/Sweatproof Wireless Headphones for iOS/Android Headphones & Earbuds    4.6  13.99
Bluetooth Headphones Over-Ear Wireless Headset in Full-Synthetic-Leather Wrapped, V5.3 Deep Bass HiFi Stereo with Build-in Microphone Headphone 3.5mm Wired Headphone 40H Playtime in Foldable Design Headphones & Earbuds    4.4  29.99
                   